### Landing to bronze 100

In [1]:
%run "ms_lms_config"

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 4, Finished, Available, Finished)

In [2]:
today_date = ''

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 5, Finished, Available, Finished)

In [3]:
partition_path = f'/Processing_Date={today_date}'


StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 6, Finished, Available, Finished)

In [4]:
landing_adls_path = f'abfss://{raw_container}@{account_name}.dfs.core.windows.net/{landing_relative_path}'
complete_landing_adls_path = landing_adls_path + partition_path

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 7, Finished, Available, Finished)

In [5]:
landing_schema = StructType([
    StructField("Student_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Grade_Level", StringType(), True),
    StructField("Course_ID", StringType(), True),
    StructField("Course_Name", StringType(), True),
    StructField("Enrollment_Date", StringType(), True),
    StructField("Completion_Date", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Final_Grade", StringType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Time_Spent_on_Course_hrs", DoubleType(), True),
    StructField("Assignments_Completed", IntegerType(), True),
    StructField("Quizzes_Completed", IntegerType(), True),
    StructField("Forum_Posts", IntegerType(), True),
    StructField("Messages_Sent", IntegerType(), True),
    StructField("Quiz_Average_Score", DoubleType(), True),
    StructField("Assignment_Scores", StringType(), True),
    StructField("Assignment_Average_Score", DoubleType(), True),
    StructField("Project_Score", DoubleType(), True),
    StructField("Extra_Credit", DoubleType(), True),
    StructField("Overall_Performance", DoubleType(), True),
    StructField("Feedback_Score", DoubleType(), True),
    StructField("Parent_Involvement", StringType(), True),
    StructField("Demographic_Group", StringType(), True),
    StructField("Internet_Access", StringType(), True),
    StructField("Learning_Disabilities", StringType(), True),
    StructField("Preferred_Learning_Style", StringType(), True),
    StructField("Language_Proficiency", StringType(), True),
    StructField("Participation_Rate", StringType(), True),
    StructField("Completion_Time_Days", IntegerType(), True),
    StructField("Performance_Score", DoubleType(), True),
    StructField("Course_Completion_Rate", DoubleType(), True)
])

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 8, Finished, Available, Finished)

In [6]:
df = spark \
.read \
.format('csv') \
.option('header','true') \
.schema(landing_schema) \
.load(path=complete_landing_adls_path)


StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 9, Finished, Available, Finished)

In [7]:
windowSpec = Window.partitionBy("Student_ID", "Course_ID").orderBy("Enrollment_Date")
deduped_source = df \
    .withColumn("rn", row_number().over(windowSpec)) \
    .filter("rn == 1") \
    .drop("rn")

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 10, Finished, Available, Finished)

In [8]:
deduped_source.createOrReplaceTempView('lms_100_bronze_view_new_data')


StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 11, Finished, Available, Finished)

### Create bronze table

In [9]:
lms_100_bronze_path = 'abfss://30b7edc1-64f9-4583-8c8f-2498c1953029@onelake.dfs.fabric.microsoft.com/4f41bc1f-118f-4766-ab52-f9ff821831d7/Tables/lms_100_bronze'

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 12, Finished, Available, Finished)

In [10]:
create_bronze_table = f"""CREATE TABLE IF NOT EXISTS lms_100_bronze (
        Student_ID STRING,
        Name STRING,
        Age INT,
        Gender STRING,
        Grade_Level STRING,
        Course_ID STRING,
        Course_Name STRING,
        Enrollment_Date STRING,
        Completion_Date STRING,
        Status STRING,
        Final_Grade STRING,
        Attendance_Rate DOUBLE,
        Time_Spent_on_Course_hrs DOUBLE,
        Assignments_Completed INT,
        Quizzes_Completed INT,
        Forum_Posts INT,
        Messages_Sent INT,
        Quiz_Average_Score DOUBLE,
        Assignment_Scores STRING,
        Assignment_Average_Score DOUBLE,
        Project_Score DOUBLE,
        Extra_Credit DOUBLE,
        Overall_Performance DOUBLE,
        Feedback_Score DOUBLE,
        Parent_Involvement STRING,
        Demographic_Group STRING,
        Internet_Access STRING,
        Learning_Disabilities STRING,
        Preferred_Learning_Style STRING,
        Language_Proficiency STRING,
        Participation_Rate STRING,
        Completion_Time_Days INT,
        Performance_Score DOUBLE,
        Course_Completion_Rate DOUBLE,
        Processing_Date DATE )"""

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 13, Finished, Available, Finished)

In [11]:
try:
    spark \
    .read \
    .format('delta') \
    .load(lms_100_bronze_path) \
    .createOrReplaceTempView('lms_100_bronze_view')
except:
    spark.sql(create_bronze_table)
    



StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 14, Finished, Available, Finished)

In [12]:
%%sql

select * from lms_100_bronze_view_new_data

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 15, Finished, Available, Finished)

<Spark SQL result set with 11 rows and 34 fields>

In [13]:
%%sql

select * from lms_100_bronze_view

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 16, Finished, Available, Finished)

<Spark SQL result set with 15 rows and 35 fields>

In [14]:

merge_lms_100_bronze = \
f"""     
MERGE INTO lms_100_bronze as TARGET
USING lms_100_bronze_view_new_data as SOURCE
ON TARGET.Student_ID = SOURCE.Student_ID AND TARGET.Course_ID = SOURCE.Course_ID

WHEN MATCHED THEN
                            UPDATE SET
                                TARGET.Name =                       SOURCE.Name,
                                TARGET.Age =                        SOURCE.Age,
                                TARGET.Gender =                     SOURCE.Gender,
                                TARGET.Grade_Level =                SOURCE.Grade_Level,
                                TARGET.Course_Name =                SOURCE.Course_Name,
                                TARGET.Enrollment_Date =            SOURCE.Enrollment_Date,
                                TARGET.Completion_Date =            SOURCE.Completion_Date,
                                TARGET.Status =                     SOURCE.Status,
                                TARGET.Final_Grade =                SOURCE.Final_Grade,
                                TARGET.Attendance_Rate =            SOURCE.Attendance_Rate,
                                TARGET.Time_Spent_on_Course_hrs =   SOURCE.Time_Spent_on_Course_hrs,
                                TARGET.Assignments_Completed =      SOURCE.Assignments_Completed,
                                TARGET.Quizzes_Completed =          SOURCE.Quizzes_Completed,
                                TARGET.Forum_Posts =                SOURCE.Forum_Posts,
                                TARGET.Messages_Sent =              SOURCE.Messages_Sent,
                                TARGET.Quiz_Average_Score =         SOURCE.Quiz_Average_Score,
                                TARGET.Assignment_Scores =          SOURCE.Assignment_Scores,
                                TARGET.Assignment_Average_Score =   SOURCE.Assignment_Average_Score,
                                TARGET.Project_Score =              SOURCE.Project_Score,
                                TARGET.Extra_Credit =               SOURCE.Extra_Credit,
                                TARGET.Overall_Performance =        SOURCE.Overall_Performance,
                                TARGET.Feedback_Score =             SOURCE.Feedback_Score,
                                TARGET.Parent_Involvement =         SOURCE.Parent_Involvement,
                                TARGET.Demographic_Group =          SOURCE.Demographic_Group,
                                TARGET.Internet_Access =            SOURCE.Internet_Access,
                                TARGET.Learning_Disabilities =      SOURCE.Learning_Disabilities,
                                TARGET.Preferred_Learning_Style =   SOURCE.Preferred_Learning_Style,
                                TARGET.Language_Proficiency =       SOURCE.Language_Proficiency,
                                TARGET.Participation_Rate =         SOURCE.Participation_Rate,
                                TARGET.Completion_Time_Days =       SOURCE.Completion_Time_Days,
                                TARGET.Performance_Score =          SOURCE.Performance_Score,
                                TARGET.Course_Completion_Rate =     SOURCE.Course_Completion_Rate,
                                TARGET.Processing_Date = '{today_date}'
WHEN NOT MATCHED THEN
                            INSERT                  
                            (
                                TARGET.Student_ID, 
                                TARGET.Name, 
                                TARGET.Age, 
                                TARGET.Gender, 
                                TARGET.Grade_Level, 
                                TARGET.Course_ID, 
                                TARGET.Course_Name, 
                                TARGET.Enrollment_Date, 
                                TARGET.Completion_Date, 
                                TARGET.Status, 
                                TARGET.Final_Grade, 
                                TARGET.Attendance_Rate, 
                                TARGET.Time_Spent_on_Course_hrs, 
                                TARGET.Assignments_Completed, 
                                TARGET.Quizzes_Completed,
                                TARGET.Forum_Posts, 
                                TARGET.Messages_Sent, 
                                TARGET.Quiz_Average_Score, 
                                TARGET.Assignment_Scores, 
                                TARGET.Assignment_Average_Score, 
                                TARGET.Project_Score,
                                TARGET.Extra_Credit, 
                                TARGET.Overall_Performance, 
                                TARGET.Feedback_Score, 
                                TARGET.Parent_Involvement, 
                                TARGET.Demographic_Group, 
                                TARGET.Internet_Access,
                                TARGET.Learning_Disabilities, 
                                TARGET.Preferred_Learning_Style, 
                                TARGET.Language_Proficiency, 
                                TARGET.Participation_Rate, 
                                TARGET.Completion_Time_Days,
                                TARGET.Performance_Score, 
                                TARGET.Course_Completion_Rate, 
                                TARGET.Processing_Date
                            )
                            VALUES 
                            (
                                SOURCE.Student_ID,
                                SOURCE.Name, 
                                SOURCE.Age, 
                                SOURCE.Gender, 
                                SOURCE.Grade_Level, 
                                SOURCE.Course_ID, 
                                SOURCE.Course_Name, 
                                SOURCE.Enrollment_Date, 
                                SOURCE.Completion_Date, 
                                SOURCE.Status, 
                                SOURCE.Final_Grade, 
                                SOURCE.Attendance_Rate, 
                                SOURCE.Time_Spent_on_Course_hrs, 
                                SOURCE.Assignments_Completed, 
                                SOURCE.Quizzes_Completed, 
                                SOURCE.Forum_Posts, 
                                SOURCE.Messages_Sent, 
                                SOURCE.Quiz_Average_Score, 
                                SOURCE.Assignment_Scores, 
                                SOURCE.Assignment_Average_Score, 
                                SOURCE.Project_Score, 
                                SOURCE.Extra_Credit, 
                                SOURCE.Overall_Performance, 
                                SOURCE.Feedback_Score, 
                                SOURCE.Parent_Involvement, 
                                SOURCE.Demographic_Group, 
                                SOURCE.Internet_Access, 
                                SOURCE.Learning_Disabilities, 
                                SOURCE.Preferred_Learning_Style, 
                                SOURCE.Language_Proficiency, 
                                SOURCE.Participation_Rate, 
                                SOURCE.Completion_Time_Days, 
                                SOURCE.Performance_Score, 
                                SOURCE.Course_Completion_Rate, '{today_date}'
                            )
"""
                        

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 17, Finished, Available, Finished)

In [15]:
spark.sql(merge_lms_100_bronze).show()

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 18, Finished, Available, Finished)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|               11|               1|               0|               10|
+-----------------+----------------+----------------+-----------------+



In [16]:
%%sql


SELECT * FROM ms_lms_lh_100_bronze.lms_100_bronze LIMIT 1000

StatementMeta(, 034b0e44-8ba8-472c-a70b-e73b5a4a1762, 19, Finished, Available, Finished)

<Spark SQL result set with 25 rows and 35 fields>